# Standards Mapping And Virtual KG

This notebook shows standards-oriented graph construction paths: R2RML for relational rows, RML for cross-source joins, CSVW for tabular metadata, JSON-LD expansion into RDF facts, and OBDA-style virtual graph query rewriting.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            ROOT = parent
            break
sys.path.insert(0, str(ROOT / "src"))


## R2RML relational rows → RDF facts

R2RML maps relational tables into RDF terms. The example below uses a referencing object map so a `manager_id` column becomes a `reportsTo` edge.


In [ ]:
from kg_construction.domain.models import SourceRecord
from kg_construction.infrastructure.mapping import (
    CSVWColumnConfig,
    CSVWDialectConfig,
    CSVWDialectParser,
    CSVWMapper,
    CSVWMappingConfig,
    CSVWTableConfig,
    CSVWTableSchemaConfig,
    JoinConditionConfig,
    OBDAPredicateMapping,
    OBDAVirtualGraphMapper,
    PredicateObjectMapConfig,
    R2RMLMapper,
    R2RMLMappingConfig,
    RMLMapper,
    RMLMappingConfig,
    TermMapConfig,
    TriplesMapConfig,
)
from kg_construction.infrastructure.mapping import JSONLDMapper, JSONLDMappingConfig

people = SourceRecord.from_rows(
    [{"id": "1", "name": "Ada"}, {"id": "2", "name": "Grace", "manager_id": "1"}],
    source_type="sql",
    source_name="memory://db#people",
    metadata={"table": "people"},
)
managers = SourceRecord.from_rows(
    [{"id": "1", "name": "Ada"}],
    source_type="sql",
    source_name="memory://db#managers",
    metadata={"table": "managers"},
)

r2rml_config = R2RMLMappingConfig(
    triples_maps=[
        TriplesMapConfig(
            mapping_id="people_map",
            logical_table="people",
            subject_map=TermMapConfig(template="urn:person:{id}"),
            predicate_object_maps=[
                PredicateObjectMapConfig(
                    predicate_map=TermMapConfig(constant="urn:kg:reportsTo"),
                    parent_triples_map="manager_map",
                    join_conditions=[JoinConditionConfig(child="manager_id", parent="id")],
                )
            ],
        ),
        TriplesMapConfig(
            mapping_id="manager_map",
            logical_table="managers",
            subject_map=TermMapConfig(template="urn:person:{id}"),
        ),
    ]
)
r2rml_batch = R2RMLMapper(r2rml_config).map_sources([people, managers])
[(fact.subject, fact.predicate, fact.object, fact.properties.get("standard")) for fact in r2rml_batch.facts]


## RML and CSVW for tabular/semi-structured inputs

RML generalizes the mapping idea beyond one relational table. CSVW adds dialect and schema metadata so CSV parsing and validation are not guessed from the file alone.


In [ ]:
csv_text = "# comment\nid;name\n1; Ada \n2;Grace"
rows = CSVWDialectParser(CSVWDialectConfig(delimiter=";", commentPrefix="#", trim=True)).parse_text(csv_text)
csv_source = SourceRecord.from_rows(rows, source_type="csv", source_name="people.csv")
csvw_table = CSVWTableConfig(
    url="people.csv",
    tableSchema=CSVWTableSchemaConfig(
        primaryKey="id",
        columns=[CSVWColumnConfig(name="id", required=True), CSVWColumnConfig(name="name", required=True)],
    ),
)
csvw_batch = CSVWMapper(CSVWMappingConfig(tables=[csvw_table])).map_source(csv_source)

rml_config = RMLMappingConfig(
    triples_maps=[
        TriplesMapConfig(
            mapping_id="people_rml",
            logical_source="people.csv",
            reference_formulation="csv",
            subject_map=TermMapConfig(template="urn:person:{id}"),
            predicate_object_maps=[
                PredicateObjectMapConfig(
                    predicate_map=TermMapConfig(constant="urn:kg:name"),
                    object_map=TermMapConfig(reference="name"),
                )
            ],
        )
    ]
)
rml_batch = RMLMapper(rml_config).map_source(csv_source)
{
    "csvw_valid": csvw_batch.metadata["csvw_validation"]["valid"],
    "csvw_fact_count": len(csvw_batch.facts),
    "rml_fact_count": len(rml_batch.facts),
}


## JSON-LD expansion and RDF dataset facts

JSON-LD input can be expanded and emitted as evidence-backed graph facts. Named graph metadata preserves construction context when RDF datasets are needed.


In [ ]:
jsonld_source = SourceRecord.from_json(
    {
        "@context": {"name": "http://schema.org/name", "serviceType": "http://schema.org/serviceType"},
        "@id": "urn:svc:recommendation",
        "name": "Recommendation API",
        "serviceType": "API",
    },
    source_name="memory://jsonld",
).model_copy(update={"source_type": "jsonld"})
jsonld_batch = JSONLDMapper(JSONLDMappingConfig(named_graph="urn:kg:graph:example")).map_source(jsonld_source)
{
    "standard": jsonld_batch.metadata["standard"],
    "facts": len(jsonld_batch.facts),
    "named_graph": jsonld_batch.metadata["rdf_dataset_graph"],
}


## OBDA virtual graph query rewriting

OBDA keeps data in a relational source and rewrites simple graph-shaped queries into SQL plans. This is useful when the graph should be queried virtually before materialization.


In [ ]:
obda = OBDAVirtualGraphMapper(
    [
        OBDAPredicateMapping(
            predicate_iri="urn:kg:name",
            table="people",
            subject_template="urn:person:{id}",
            object_column="name",
        )
    ]
)
from dataclasses import asdict

plans = obda.rewrite_select("SELECT ?s ?o WHERE { ?s <urn:kg:name> ?o . }")
[asdict(plan) for plan in plans]
